# Analysis parameters

In [ ]:
import re
import time

import nibabel as nib
import numpy as np
import colorcet as cc
import pandas as pd

import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib

from nilearn.masking import apply_mask, unmask
from nilearn.image import math_img, high_variance_confounds

# Reportlab for pdf 

from reportlab.pdfbase import pdfmetrics
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

In [2]:
# Reportlab params for reports

MontSerratBold = pdfmetrics.EmbeddedType1Face('../../../utils/fonts/montserrat/Montserrat-Bold.afm', '../../../utils/fonts/montserrat/Montserrat-Bold.pfb')
MontSerratLight = pdfmetrics.EmbeddedType1Face('../../../utils/fonts/montserrat/Montserrat-Light.afm', '../../../utils/fonts/montserrat/Montserrat-Light.pfb')

# Montserrat Bold

pdfmetrics.registerTypeFace(MontSerratBold)
montserrat_bold_name = 'Montserrat-Bold'

montserrat_bold_font = pdfmetrics.Font('Montserrat-Bold',
                           montserrat_bold_name,
                           'WinAnsiEncoding')

pdfmetrics.registerFont(montserrat_bold_font)

# Montserrat Light

pdfmetrics.registerTypeFace(MontSerratLight)
montserrat_light_name = 'Montserrat-Light'

montserrat_light_font = pdfmetrics.Font('Montserrat-Light',
                           montserrat_light_name,
                           'WinAnsiEncoding')

pdfmetrics.registerFont(montserrat_light_font)

# Helper functions

In [68]:
def standardize(signals):
    
    with np.errstate(divide = 'ignore', invalid = 'ignore'):
        signals_norm = (signals - signals.mean(0))/signals.std(0)
        
    signals_norm[np.isnan(signals_norm)] = 0
    
    return signals_norm

def write_regressor_txt(data, file_name, nb_timestamp = None):
    
    if data.squeeze().shape[0]==0:
        
        with open(file_name, 'w') as f:

            for i in range(nb_timestamp):
                
                f.writelines(['1.0\n'])
    
    elif len(data.shape) == 1:

        with open(file_name, 'w') as f:

            for x in data:

                f.writelines([str(x) + '\t1.0\n'])
                
    else:
        
        with open(file_name, 'w') as f:

            for x in data:

                f.writelines('\t'.join([str(y) for y in x]) + '\t1.0\n')
            
    return

def process_regressors(
    file_name,
    mask_list,
    wmcsf_list
):

    ###### Gets file features

    subject_name = 'sub-' + file_name.name.split('sub-')[1].split('_')[0]
    session_name = 'ses-' + file_name.name.split('ses-')[1].split('_')[0]
    pose_id = 'pose-' + file_name.name.split('pose-')[1].split('_')[0]
    pose_idx = int(pose_id[-1])
    if 'run' in file_name.name:
        run_id = 'run-' + file_name.name.split('run-')[1].split('_')[0]
    
    prefix = file_name.name.split('_pwd')[0]
    suffix = file_name.name.split('_')[-1]
    
    save_folder = save_dir / subject_name  / session_name 
    
    ###### Loads data
    
    img = nib.load(file_name)
    [X,Y,Z,T] = img.shape
    
    ###### compute regressors
    
    gs, fmr, confounds_1_2, confounds_1_100, confounds_5_2, low_intensity_signal, wmcsf_signal, acompcor, no_reg, out_PC5 = compute_regressors(
        img, 
        mask_list, 
        wmcsf_list,
        pose_idx    
    )
    
    ###### OUTPUT
    
    suffixes = ['GlobalSignal', 'FirstMode', 'tCompCorN1P2', 'tCompCorN1P100', 'tCompCorN5P2', 'lowIntensitySignal', 'WMCSFsignal', 'aCompCor', 'NoReg', 'outPC5']
    signals = [gs.squeeze(), fmr.squeeze(), confounds_1_2.squeeze(), confounds_1_100.squeeze(), confounds_5_2.squeeze(), low_intensity_signal.squeeze(), wmcsf_signal.squeeze(), acompcor.squeeze(), no_reg.squeeze(), out_PC5.squeeze()]
    nb_timestamps = [None, None,None,None,None,None,None,None,len(wmcsf_signal),None]
    
    # exports regressors
            
    for data, suffix, nb_timestamp in zip(signals, suffixes, nb_timestamps):
    
        save_name = save_folder / (file_name.name.split('_pwd')[0]+f'_regressor-{suffix}.txt')
        save_name.parent.mkdir(exist_ok = True, parents = True)
        write_regressor_txt(data, save_name, nb_timestamp = nb_timestamp)

    return gs, fmr, confounds_1_2, confounds_1_100, confounds_5_2, low_intensity_signal, wmcsf_signal, acompcor, no_reg, out_PC5


def compute_regressors(
    img, 
    mask_list, 
    wmcsf_list,
    pose_idx    
):
    
    signals_norm = apply_mask(img, mask_list[pose_idx])
    
    # GS
    
    gs = standardize(standardize(signals_norm).mean(1)[:,None])
    
    # FM
    
    [u,s,v] = np.linalg.svd(signals_norm.T)
    fmr = standardize(v[0,:][:,None])
    
    # tcompcor
    
    confounds_5_2 = high_variance_confounds(
        img, 
        n_confounds=5, 
        percentile=2.0, 
        detrend=True, 
        mask_img=mask_list[pose_idx]
    )
    
    confounds_1_2 = high_variance_confounds(
        img, 
        n_confounds=1, 
        percentile=2.0, 
        detrend=True, 
        mask_img=mask_list[pose_idx]
    )
    
    confounds_1_100 = high_variance_confounds(
        img, 
        n_confounds=1, 
        percentile=100.0, 
        detrend=True, 
        mask_img=mask_list[pose_idx]
    )
    
    # WM/CSF
    
    wmcsf_signal = standardize(apply_mask(img, wmcsf_list[pose_idx])).mean(1)
    
    # aCompCor
    
    acompcor = high_variance_confounds(
        img, 
        n_confounds=5, 
        percentile=100, 
        mask_img=wmcsf_list[pose_idx]
    )
    
    # No Regression 
    
    no_reg = np.array([1.0]*img.shape[-1])
    
    # out PC5
    
    out_PC5 = high_variance_confounds(
        img, 
        n_confounds=5,
        percentile=100,
        mask_img=mask_out_list[pose_idx]
    )
    
    # gets the non skull stripped file (for low intensity signal)
    
    average_img = apply_mask(img, mask_list[pose_idx]).mean(0)
    
    low_intensity_vect = (average_img < np.percentile(average_img, 5.)).astype('uint32')
    
    low_intensity_img = unmask(low_intensity_vect, mask_list[pose_idx])
    
    low_intensity_signal = standardize(standardize(apply_mask(img, low_intensity_img)).mean(1)[:,None])

    return gs, fmr, confounds_1_2, confounds_1_100, confounds_5_2, low_intensity_signal, wmcsf_signal, acompcor, no_reg, out_PC5

# General Parameters

In [12]:
params_root = Path('/media/DATA2/JC/1_DATA/2025-02-24_PepeMariani/derivatives/01_SWregistration/derivatives/Params/pose_templates/')

mask_dir = sorted([x for x in params_root.iterdir() if x.is_file() and re.match(f'^.*AllenMask.*pose-(0|1|2|3).*.nii.gz$', x.name)])

mask_list = [nib.load(x) for x in mask_dir]
mask_out_list = [math_img("a==0", a = nib.load(x)) for x in mask_dir]

wmcsf_dir = sorted([x for x in params_root.iterdir() if x.is_file() and re.match(f'^.*WMCSFmask.*pose-(0|1|2|3).*.nii.gz$', x.name)])

wmcsf_list = [nib.load(x) for x in wmcsf_dir]

for x in wmcsf_dir:

    print(x.name)

source-Gozzilab_space-fUSIswVBm40_desc-WMCSFmask_res-110umx110umx100um_pose-0_feature.nii.gz
source-Gozzilab_space-fUSIswVBm40_desc-WMCSFmask_res-110umx110umx100um_pose-1_feature.nii.gz
source-Gozzilab_space-fUSIswVBm40_desc-WMCSFmask_res-110umx110umx100um_pose-2_feature.nii.gz
source-Gozzilab_space-fUSIswVBm40_desc-WMCSFmask_res-110umx110umx100um_pose-3_feature.nii.gz


# Regressors

## Parameters

In [53]:
root_path = Path('/media/DATA2/JC/1_DATA/2025-02-24_PepeMariani/derivatives/01_SWregistration/derivatives/01_avg-dB-registered/')

root_dir = sorted([x for x in root_path.rglob('*') if x.is_file() and re.match(f'^.*proc-choppedSWregistered.*.nii.gz$', x.name)])

print(f'We found {len(root_dir)} files')

t_r = 2.4

save_dir = Path('/media/DATA2/JC/1_DATA/2025-02-24_PepeMariani/derivatives/02_preprocessing/derivatives/regressors')
save_dir.mkdir(parents = True, exist_ok = True)

We found 210 files


## Loop

In [75]:
for file_name in tqdm(root_dir):

    _ = process_regressors(
        file_name,
        mask_list,
        wmcsf_list
    )
    

100%|████████████████████████████| 210/210 [4:03:59<00:00, 69.71s/it]
